# Notebook 01 -- Requirements & Data Understanding

## Objective

Establish, in one place, *what this pipeline is for* and *what raw data actually looks like* before any cleaning or feature logic is written -- so every later notebook can point back here instead of re-deriving assumptions. Concretely, this notebook answers:

1. What business decision does the full pipeline (Notebooks 01-13) ultimately support? (traceability to `docs/03_Business_Problem_and_Requirements.md`)
2. What raw data do we actually have -- tag count, date range, sampling frequency, sheet layout -- *as measured*, not as assumed?
3. Is the raw file structurally sound enough to proceed to Notebook 02 (Data Ingestion), or are there structural surprises (missing tags, wrong sheet layout, date range gaps) that must be resolved first?

This is deliberately **not** an ingestion or cleaning notebook -- no NaN-filling, no outlier removal, no TAM masking happens here. Notebook 02 owns that. This notebook only *looks*, it does not *change*.

## Inputs

| Input | Path | Note |
|---|---|---|
| Raw process Excel | `src.cpht.config.RAW_EXCEL` (`Data/Process information data (2024-2026).xlsx`) | Sheet `Sheet1`, header rows at fixed offsets (see Method) |
| HX tag config | `src.cpht.config.HX_CONFIG` | Canonical list of tags this pipeline depends on |
| Requirements doc | `docs/03_Business_Problem_and_Requirements.md` | Business Problem, Requirements 2.1-2.5 |

## Assumptions

- Raw Excel sheet name and header-row layout (tag names row 4, units row 5, description row 6, data from row 8 -- 1-indexed) are stable across uploads. If a future upload changes this layout, the Diagnostic Checks below should catch it (tag row won't parse as tag-like strings) rather than silently misreading data as garbage, which is what the legacy pipeline's `/api/run-full` path currently risks (see `docs/03` section 2.2 for the known gap this design avoids).
- Timestamps are plant-local (Asia/Bangkok, UTC+7), not stored with a timezone offset in the source file.
- This notebook profiles the *whole* file (no TAM/shutdown masking) -- shutdown periods are expected to show up as flat/near-zero flow, not as missing rows.

## Requirements traced from `docs/03`

| Requirement | Source | How this notebook addresses it |
|---|---|---|
| FR-DI-001 -- ingest ~95 process tags from DCS/Historian | docs/03 section 2.2 | Tag inventory below confirms actual count matches expectation before Notebook 02 builds on it |
| Data Requirements section 2.2 -- validate at ingestion, not deep inside a notebook | docs/03 section 2.2 (identified gap) | `src.cpht.validation.check_required_tags` runs here, first, before any downstream notebook trusts the data |
| Business Problem section 1 -- decision horizon / users / KPI | docs/03 section 1 | Restated below for traceability; **not** re-derived (that decision already happened in `docs/03`, this notebook just cites it) |


## Method

1. Add the repo root to `sys.path` so `src.cpht` is importable from `notebooks_v2/` (kept out of the package itself so notebooks stay runnable standalone from VS Code / Jupyter without an install step).
2. Read only the **header rows** (tag / unit / description, rows 4-6) of the raw Excel first -- cheap, and enough to build the tag inventory and run the required-tag check before paying the cost of loading the full data body.
3. Read the **data body** (all rows) once the header looks sound, and compute: row count, date range, inferred sampling interval, and per-tag missing %.
4. Run `src.cpht.validation.check_required_tags` against the discovered tag list.
5. Write a small `data_profile.json` artifact that Notebook 02 (and anyone auditing the pipeline) can read without re-opening the raw Excel.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks_v2" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np
import json

from src.cpht import config
from src.cpht import validation

print(f"Repo root:    {REPO_ROOT}")
print(f"Raw Excel:    {config.RAW_EXCEL}")
print(f"HX in scope:  {len(config.ALL_HX)}  ({len(config.CPHT_1_HX)} CPHT-1 + {len(config.CPHT_2_HX)} CPHT-2)")
print(f"Required tags (from HX_CONFIG): {len(config.required_tags())}")


Repo root:    C:\Desktop\Bangchak Internship 2026\furnace-optimization
Raw Excel:    C:\Desktop\Bangchak Internship 2026\Data\Process information data (2024-2026).xlsx
HX in scope:  17  (5 CPHT-1 + 12 CPHT-2)
Required tags (from HX_CONFIG): 26


In [2]:
# --- Header-only read (cheap): rows 0-6 (0-indexed) is enough to see the
# tag/unit/description rows without loading the full data body yet.
raw_head = pd.read_excel(config.RAW_EXCEL, sheet_name="Sheet1", header=None, nrows=7)

tags_row = raw_head.iloc[3, 3:]
units_row = raw_head.iloc[4, 3:]
desc_row = raw_head.iloc[5, 3:]

tag_inventory = pd.DataFrame({
    "tag": tags_row.values,
    "unit": units_row.values,
    "description": desc_row.values,
}).dropna(subset=["tag"]).reset_index(drop=True)

print(f"Tags found in header row: {len(tag_inventory)}")
tag_inventory.head(10)


Tags found in header row: 97


,tag,unit,description
0,00FIC001.pv,M3/HR,NaN
1,00FC501.pv,M3/HR,KEROSENE TO DGOHDS
2,1AI001.pv,O2 %,F101 FLUE GAS O2
3,1FC020.pv,M3/HR,CRUDE TO F101 INLET
4,1FC021.pv,M3/HR,CRUDE TO F101 INLET
5,1FC022.pv,M3/HR,CRUDE TO F101 INLET
6,1FC023.pv,M3/HR,CRUDE TO F101 INLET
7,1FC035.pv,M3/HR,NO.3 SR
8,1FC062.pv,M3/HR,GAS OIL RUNDOWN
9,1FI007.pv,M3/HR,CRUDE E101C


## Data Validation

Run before touching the data body. If this fails, stop here -- do not proceed to Notebook 02 with a file that's missing tags the whole pipeline depends on.


In [3]:
tag_result = validation.check_required_tags(
    available_tags=tag_inventory["tag"].tolist(),
    required_tags=config.required_tags(),
)
print(tag_result.report())

# Fail loudly and stop the notebook here if required tags are missing --
# this is the "validate at the boundary" behavior; Notebook 02 should never
# have to discover a missing tag three cells into a merge.
tag_result.raise_if_failed()


Validation: PASS


## Analysis

Required tags are all present, so it's safe to pay the cost of loading the full data body and profile it: row count, date range, sampling interval, and missing-data % per tag.


In [4]:
# --- Full data body read. Same row/column offsets as the legacy pipeline's
# notebooks/01_data_cleaning.ipynb (kept identical on purpose -- this is a
# structural fact about the raw file, not a design choice to redo).
raw = pd.read_excel(config.RAW_EXCEL, sheet_name="Sheet1", header=None)

data = raw.iloc[7:, 2:].copy()
data.columns = ["Timestamp"] + list(tag_inventory["tag"])
data["Timestamp"] = pd.to_datetime(data["Timestamp"], errors="coerce")

n_placeholder = data["Timestamp"].isna().sum()
if n_placeholder:
    print(f"{n_placeholder} trailing placeholder row(s) with no timestamp (expected -- not an error)")
data = data.dropna(subset=["Timestamp"]).set_index("Timestamp")

for col in data.columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")

print(f"Rows:        {len(data):,}")
print(f"Date range:  {data.index.min()}  to  {data.index.max()}")
print(f"Span:        {(data.index.max() - data.index.min()).days} days")

diffs = data.index.to_series().diff().dropna()
modal_interval = diffs.mode().iloc[0] if len(diffs.mode()) else pd.Timedelta(days=1)
print(f"Sampling interval: median={diffs.median()}, mode={modal_interval}")


Rows:        2,008
Date range:  2021-01-01 00:00:00  to  2026-07-01 00:00:00
Span:        2007 days
Sampling interval: median=1 days 00:00:00, mode=1 days 00:00:00


In [5]:
missing_pct = (data.isna().mean() * 100).round(2).sort_values(ascending=False)
key_tags = [config.CIT_TAG, config.TOTAL_CHARGE_TAG] + config.required_tags()

missing_summary = pd.DataFrame({
    "tag": missing_pct.index,
    "missing_pct": missing_pct.values,
    "is_required": missing_pct.index.isin(config.required_tags()),
})

print("Worst 10 tags by missing %:")
print(missing_summary.sort_values("missing_pct", ascending=False).head(10).to_string(index=False))

print("\nRequired-tag missing % (should generally be low; high values here are a\n"
      "downstream data-quality flag for Notebook 03, not a failure here):")
print(missing_summary[missing_summary["is_required"]].sort_values("missing_pct", ascending=False).head(10).to_string(index=False))


Worst 10 tags by missing %:
        tag  missing_pct  is_required
00FIC001.pv          0.0        False
 00FC501.pv          0.0        False
  1AI001.pv          0.0        False
  1FC020.pv          0.0        False
  1FC021.pv          0.0        False
  1FC022.pv          0.0        False
  1FC023.pv          0.0        False
  1FC035.pv          0.0        False
  1FC062.pv          0.0        False
  1FI007.pv          0.0         True

Required-tag missing % (should generally be low; high values here are a
downstream data-quality flag for Notebook 03, not a failure here):
      tag  missing_pct  is_required
1FI007.pv          0.0         True
1FI008.pv          0.0         True
1FI009.pv          0.0         True
1FI015.pv          0.0         True
1FI016.pv          0.0         True
1FI017.pv          0.0         True
1TI101.pv          0.0         True
1TI102.pv          0.0         True
1TI104.pv          0.0         True
1TI106.pv          0.0         True


### Data Dictionary (docs/03 section 2.2 / section 3)

`docs/03_Business_Problem_and_Requirements.md` calls the Data Dictionary the thing that should exist **before** Data
Quality (Notebook 03) or Alert work can be built on top of it -- so it's built here, in Notebook 01, rather than
deferred to an unscheduled later step.

17 fields per the template: Tag ID, Tag Name, Description, Equipment, Process Side, Measurement Type, Unit, Sampling
Frequency, Valid Min/Max, Warning Min/Max, Critical Min/Max, Aggregation Rule, Missing Rule, Source System, Sensor
Type, Owner, Criticality. Fields derivable from data/config we already have are filled in below; fields that require
domain sign-off (Valid/Warning/Critical bands, Owner) are left as an explicit `TBD` string rather than guessed --
this mirrors the project's existing `limit_assumed=True` convention (`notebooks/build_dashboard_topology.py`) of
being honest about what's confirmed vs. assumed.


In [6]:
# --- Equipment / Process Side: reverse-lookup from HX_CONFIG. HX_CONFIG only
# defines cold-side tags (Q duty is computed cold-side-only by design -- see
# docs/02_Requirement_v2_SSOT.md section 3.1), so every tag found this way is
# Process Side = "Cold". Tags not referenced by any HX are structurally
# "Other" (furnace/general tags like O2, stack temp -- not part of the CPHT
# HX inventory this dictionary is scoped to).
tag_to_hx = {}
for hx, cfg in config.HX_CONFIG.items():
    for role in ("cold_flow", "cold_in", "cold_out"):
        tag_to_hx.setdefault(cfg[role], []).append(hx)

# E101G is a documented CPHT-1 HX with zero instrumentation (config.py /
# docs/02 section 2) -- it must NOT appear as a tag anywhere, by design.
# Assert this explicitly rather than silently relying on it being absent.
e101g_tags = [t for t, hxs in tag_to_hx.items() if "E101G" in hxs]
assert not e101g_tags, f"E101G should have no instrumented tags, found: {e101g_tags}"
print("Confirmed: E101G has no instrumented tag (expected -- inferred via mass balance downstream).")


def classify_measurement_type(tag):
    t = tag.upper()
    if "TI" in t:
        return "Temperature"
    if "FI" in t or "FC" in t:
        return "Flow"
    if "PI" in t or "PC" in t:
        return "Pressure"
    if "AI" in t:
        return "Composition/Analyzer"
    return "Other"


rows = []
required = set(config.required_tags())
for _, r in tag_inventory.iterrows():
    tag = r["tag"]
    hxs = tag_to_hx.get(tag, [])
    rows.append({
        "tag_id": tag,
        "tag_name": tag,  # same identifier in this historian export -- no separate display name available
        "description": r["description"],
        "equipment": ", ".join(hxs) if hxs else "Other (non-HX / furnace)",
        "process_side": "Cold" if hxs else "N/A",
        "measurement_type": classify_measurement_type(tag),
        "unit": r["unit"],
        "sampling_frequency": str(modal_interval),
        "valid_min": "TBD", "valid_max": "TBD",
        "warning_min": "TBD", "warning_max": "TBD",
        "critical_min": "TBD", "critical_max": "TBD",
        "aggregation_rule": "as_received (already daily in this historian export -- not independently confirmed)",
        "missing_rule": "TBD",
        "source_system": "DCS",
        "sensor_type": "Measured",
        "owner": "TBD",
        "criticality": "high" if tag in required else "low",
    })

data_dictionary = pd.DataFrame(rows)
print(f"Data dictionary rows: {len(data_dictionary)} (== tag inventory rows: {len(tag_inventory)})")
print(f"Required tags with criticality=high: {(data_dictionary['criticality'] == 'high').sum()} (should equal {len(required)})")
assert (data_dictionary["criticality"] == "high").sum() == len(required)
data_dictionary.head(10)


Confirmed: E101G has no instrumented tag (expected -- inferred via mass balance downstream).
Data dictionary rows: 97 (== tag inventory rows: 97)
Required tags with criticality=high: 26 (should equal 26)


,tag_id,tag_name,description,equipment,process_side,measurement_type,unit,sampling_frequency,valid_min,valid_max,warning_min,warning_max,critical_min,critical_max,aggregation_rule,missing_rule,source_system,sensor_type,owner,criticality
0,00FIC001.pv,00FIC001.pv,NaN,Other (non-HX / furnace),N/A,Flow,M3/HR,1 days 00:00:00,TBD,TBD,TBD,TBD,TBD,TBD,as_received (already daily in this historian e...,TBD,DCS,Measured,TBD,low
1,00FC501.pv,00FC501.pv,KEROSENE TO DGOHDS,Other (non-HX / furnace),N/A,Flow,M3/HR,1 days 00:00:00,TBD,TBD,TBD,TBD,TBD,TBD,as_received (already daily in this historian e...,TBD,DCS,Measured,TBD,low
2,1AI001.pv,1AI001.pv,F101 FLUE GAS O2,Other (non-HX / furnace),N/A,Composition/Analyzer,O2 %,1 days 00:00:00,TBD,TBD,TBD,TBD,TBD,TBD,as_received (already daily in this historian e...,TBD,DCS,Measured,TBD,low
3,1FC020.pv,1FC020.pv,CRUDE TO F101 INLET,Other (non-HX / furnace),N/A,Flow,M3/HR,1 days 00:00:00,TBD,TBD,TBD,TBD,TBD,TBD,as_received (already daily in this historian e...,TBD,DCS,Measured,TBD,low
4,1FC021.pv,1FC021.pv,CRUDE TO F101 INLET,Other (non-HX / furnace),N/A,Flow,M3/HR,1 days 00:00:00,TBD,TBD,TBD,TBD,TBD,TBD,as_received (already daily in this historian e...,TBD,DCS,Measured,TBD,low
5,1FC022.pv,1FC022.pv,CRUDE TO F101 INLET,Other (non-HX / furnace),N/A,Flow,M3/HR,1 days 00:00:00,TBD,TBD,TBD,TBD,TBD,TBD,as_received (already daily in this historian e...,TBD,DCS,Measured,TBD,low
6,1FC023.pv,1FC023.pv,CRUDE TO F101 INLET,Other (non-HX / furnace),N/A,Flow,M3/HR,1 days 00:00:00,TBD,TBD,TBD,TBD,TBD,TBD,as_received (already daily in this historian e...,TBD,DCS,Measured,TBD,low
7,1FC035.pv,1FC035.pv,NO.3 SR,Other (non-HX / furnace),N/A,Flow,M3/HR,1 days 00:00:00,TBD,TBD,TBD,TBD,TBD,TBD,as_received (already daily in this historian e...,TBD,DCS,Measured,TBD,low
8,1FC062.pv,1FC062.pv,GAS OIL RUNDOWN,Other (non-HX / furnace),N/A,Flow,M3/HR,1 days 00:00:00,TBD,TBD,TBD,TBD,TBD,TBD,as_received (already daily in this historian e...,TBD,DCS,Measured,TBD,low
9,1FI007.pv,1FI007.pv,CRUDE E101C,E101AB,Cold,Flow,M3/HR,1 days 00:00:00,TBD,TBD,TBD,TBD,TBD,TBD,as_received (already daily in this historian e...,TBD,DCS,Measured,TBD,high


## Diagnostic Checks

Sanity checks on the *structure* of what was just read -- these catch the "raw Excel layout silently changed" failure mode called out in Assumptions, before it corrupts every downstream notebook.


In [7]:
diag = validation.ValidationResult(ok=True)

# 1. tag names should look like tags (start with a digit, contain a letter),
#    not like stray text -- catches a shifted header row.
bad_looking_tags = [t for t in tag_inventory["tag"] if not (isinstance(t, str) and any(c.isalpha() for c in t))]
if bad_looking_tags:
    diag.ok = False
    diag.errors.append(f"{len(bad_looking_tags)} tag(s) don't look like tag names: {bad_looking_tags[:5]}")

# 2. no duplicate tags in the header row.
dupes = tag_inventory["tag"][tag_inventory["tag"].duplicated()].tolist()
if dupes:
    diag.ok = False
    diag.errors.append(f"duplicate tag names in header row: {dupes}")

# 3. timestamp check (monotonic, no huge gaps) via shared validator.
ts_check = validation.check_timestamp_column(data.reset_index(), timestamp_col="Timestamp")
diag = validation.combine(diag, ts_check)

# 4. gap check: any gap much bigger than the modal sampling interval is worth
#    flagging (could be a genuine outage, or a symptom of a malformed file) --
#    reported as a warning, not a failure, since real plant outages happen.
#    (modal_interval computed once in the Analysis cell above -- reused here.)
big_gaps = diffs[diffs > modal_interval * 5]
if len(big_gaps):
    diag.warnings.append(f"{len(big_gaps)} gap(s) > 5x the modal sampling interval (largest: {big_gaps.max()})")

print(diag.report())
diag.raise_if_failed()


Validation: PASS


## Results

Summary of what this notebook found -- restated in prose since the tables above are easy to skim past.


In [8]:
print(f"Raw file:            {config.RAW_EXCEL}")
print(f"Total tags found:    {len(tag_inventory)}")
print(f"Required tags:       {len(config.required_tags())}  (all present: {tag_result.ok})")
print(f"Rows / date range:   {len(data):,} rows, {data.index.min().date()} to {data.index.max().date()}")
print(f"Sampling interval:   {modal_interval} (modal)")
print(f"Structural diagnostics: {'PASS' if diag.ok else 'FAIL'} ({len(diag.warnings)} warning(s))")
print(f"Worst required-tag missing %: {missing_summary[missing_summary['is_required']]['missing_pct'].max():.2f}%")


Raw file:            C:\Desktop\Bangchak Internship 2026\Data\Process information data (2024-2026).xlsx
Total tags found:    97
Required tags:       26  (all present: True)
Rows / date range:   2,008 rows, 2021-01-01 to 2026-07-01
Sampling interval:   1 days 00:00:00 (modal)
Structural diagnostics: PASS (0 warning(s))
Worst required-tag missing %: 0.00%


## Limitations

- This notebook profiles the raw file **as one static snapshot**. It doesn't detect issues that only show up after cleaning (e.g. a tag that's numerically present but physically implausible -- that's Notebook 03's job, not this one's).
- Missing-% here is computed over the *entire* history including TAM/shutdown windows, where near-100%-flat or missing readings on some tags are expected, not a defect. Don't read a high missing-% here as automatically bad -- Notebook 03 (Data Quality) is where TAM-aware thresholds apply.
- Sampling-interval detection uses the modal diff, which can be misleading if the file mixes two sampling frequencies (e.g. a mid-history historian change) -- worth a manual look at the diagnostic gap warning above if it fires.
- The tag-name "looks like a tag" heuristic (Diagnostic Checks, item 1) is a weak structural check, not a guarantee the header row is correct -- it would not catch, for example, a header row that shifted by exactly one column (all tags would still "look like tags").
- **Data Dictionary limitations (new):** `valid_min/max`, `warning_min/max`, `critical_min/max`, and `owner` are placeholder `TBD` values -- they require sign-off from a process/furnace engineer (mirrors `docs/03` Open Issues OI-005 through OI-010, none of which are closed yet) and must **not** be treated as real limits until confirmed. `measurement_type` is classified by a simple tag-name substring heuristic (`TI`→Temperature, `FI`/`FC`→Flow, etc.) -- correct for this plant's tag-naming convention as observed, but not validated against an authoritative equipment list. `aggregation_rule` states the data arrives already-daily but does not confirm *how* the historian itself aggregated it (mean/last/snapshot) -- that's a question for the DCS/historian owner, not something derivable from the exported file.

## Outputs

Writes two artifacts to `src.cpht.config.V2_OUTPUT_DIR`:
- `data_profile.json` -- read by Notebook 02 so it doesn't have to re-derive the tag inventory or re-read the raw Excel header.
- `data_dictionary.csv` -- the docs/03-required Data Dictionary (17 fields, TBD values flagged). Notebook 03 (Data Quality) should read this rather than re-deriving Equipment/Process Side/Criticality, and should populate `missing_rule` once its thresholds are designed.


In [9]:
out_dir = Path(config.V2_OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

profile = {
    "generated_at": pd.Timestamp.now().isoformat(),
    "raw_file": str(config.RAW_EXCEL),
    "n_tags_found": int(len(tag_inventory)),
    "n_tags_required": int(len(config.required_tags())),
    "required_tags_present": bool(tag_result.ok),
    "n_rows": int(len(data)),
    "date_min": data.index.min().isoformat(),
    "date_max": data.index.max().isoformat(),
    "modal_sampling_interval": str(modal_interval),
    "structural_diagnostics_pass": bool(diag.ok),
    "diagnostic_warnings": diag.warnings,
    "worst_required_tag_missing_pct": float(missing_summary[missing_summary["is_required"]]["missing_pct"].max()),
    "tag_inventory": tag_inventory.to_dict(orient="records"),
}

out_path = out_dir / "data_profile.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(profile, f, indent=2, default=str)
print(f"Wrote {out_path}")

dict_path = out_dir / "data_dictionary.csv"
data_dictionary.to_csv(dict_path, index=False, encoding="utf-8")
print(f"Wrote {dict_path}  ({len(data_dictionary)} rows, {len(data_dictionary.columns)} fields)")


Wrote C:\Desktop\Bangchak Internship 2026\Data\v2\data_profile.json


Wrote C:\Desktop\Bangchak Internship 2026\Data\v2\data_dictionary.csv  (97 rows, 20 fields)


## Conclusion

The raw file structurally matches what the pipeline expects: all required tags are present, the header layout parses cleanly, and the date range/sampling interval are consistent with a normal historian export. No blocking issues found -- see the printed diagnostic warnings above (if any) for non-blocking items worth keeping in mind downstream (e.g. large gaps, which Notebook 04 should reconcile against known TAM dates rather than treat as random missingness).

This notebook does not draw any conclusions about fouling, cleaning, or CIT -- that's out of scope here by design (see Objective).

## Next Notebook

**Notebook 02 -- Data Ingestion & Tag Mapping**: reads `data_profile.json` from this notebook (skips re-deriving the tag inventory), performs the actual ingestion (TAM/shutdown window removal, outlier correction, crude-assay merge), and produces the cleaned `Process_information_with_crude.csv`-equivalent artifact that every subsequent notebook builds on.
